# Earnalism stills job `stills-pilot7-f01s014`

**Prepared, not run.** Input zip SHA256 `4077edb0bf9a749170bc2bc28ce5661472ae51b97d945a2f33f6b5c314945325`. Set `RUN = True` in the first cell, then run all. With the R2 exchange (`exchange.json` in `/content/drive/MyDrive/earnalism-audiovisual/bicharak/stills-pilot7-f01s014/`) the notebook downloads `inputs.zip` itself and uploads the outputs when it finishes; without it, copy `inputs.zip` into that Drive folder first. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 2097 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'stills-pilot7-f01s014'
TITLE = 'bicharak'
KIND = 'stills'
INPUT_ZIP_SHA256 = '4077edb0bf9a749170bc2bc28ce5661472ae51b97d945a2f33f6b5c314945325'
MODEL_REPO = 'black-forest-labs/FLUX.1-schnell'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 4
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/bicharak/stills-pilot7-f01s014'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 2097
MANIFEST_RESERVE_S = 60
PACKAGES = ('diffusers', 'transformers', 'accelerate', 'sentencepiece', 'protobuf', 'bitsandbytes', 'torch', 'mediapipe')
EXCHANGE_SCHEMA = 'earnalism.colab-exchange.v1'
# The GPU cell stops before any download below this much GPU memory (0: any GPU)
MIN_VRAM_GIB = 0


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import shutil, subprocess
# A GPU first: a CPU-only runtime would install, download and then fail with a confusing CUDA error.
_smi = shutil.which("nvidia-smi")
if not _smi or subprocess.run([_smi, "-L"], capture_output=True).returncode != 0:
    raise RuntimeError("No GPU attached: Runtime -> Disconnect and delete runtime, then Runtime -> "
                       "Change runtime type -> A100 GPU (or L4), Save, Connect, and Run all again")
print(subprocess.run([_smi, "-L"], capture_output=True, text=True).stdout.strip())
_min_gib = globals().get("MIN_VRAM_GIB", 0)
if _min_gib:  # the job's spec min_vram_gib: refuse a smaller GPU before any install or download
    _q = subprocess.run([_smi, "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True)
    try:
        _gib = int(_q.stdout.split()[0]) / 1024  # MiB of the first GPU
    except (IndexError, ValueError):
        raise RuntimeError("could not read the GPU memory from nvidia-smi (" + repr(_q.stdout)
                           + "); this job needs at least %d GiB" % _min_gib) from None
    print("GPU memory %.1f GiB (this job needs at least %d GiB)" % (_gib, _min_gib))
    if _gib < _min_gib:
        raise RuntimeError("This job needs at least %d GiB of GPU memory and this GPU has %.1f GiB: it "
                           "needs an A100 or larger: Runtime -> Change runtime type -> A100, Save, "
                           "Connect, and Run all again (nothing was downloaded)" % (_min_gib, _gib))


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
_token, _secret_problem = None, None
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN") or None
except Exception as _e:  # no secret, notebook access not granted, or the prompt timed out
    _secret_problem = type(_e).__name__
    _token = os.environ.get("HF_TOKEN") or None
if _token:
    os.environ["HF_TOKEN"] = _token
from huggingface_hub import HfApi
try:
    HfApi().auth_check(MODEL_REPO, token=_token)
except Exception as _e:
    _fix = ("accept the model's terms at https://huggingface.co/" + MODEL_REPO + " with your Hugging "
            "Face account, put a Read token from https://huggingface.co/settings/tokens in the Colab "
            "secret HF_TOKEN (key icon in the left sidebar, Notebook access on), then Runtime > "
            "Restart session and Run all")
    if _secret_problem:
        _fix += " (the HF_TOKEN secret could not be read: %s; grant access if Colab asks)" % _secret_problem
    raise RuntimeError("No access to " + MODEL_REPO + " (" + type(_e).__name__ + "): " + _fix) from None
print("model access OK:", MODEL_REPO, "(token from HF_TOKEN)" if _token else "(public)")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
# R2 exchange (references/colab-exchange.md): exchange.json in this Drive folder carries two
# short-lived single-object URLs (GET inputs.zip, PUT outputs.tar). They are capabilities: never
# print them, never copy them elsewhere.
EXCHANGE = None
if (OUT_DIR / "exchange.json").is_file():
    import time, urllib.request
    from datetime import datetime
    EXCHANGE = json.loads((OUT_DIR / "exchange.json").read_text())
    if EXCHANGE.get("schema") != EXCHANGE_SCHEMA or EXCHANGE.get("job_id") != JOB_ID:
        raise RuntimeError("exchange.json belongs to a different job; ask Claude for this job's exchange.json")
    _left = datetime.fromisoformat(EXCHANGE["expires_at"].replace("Z", "+00:00")).timestamp() - time.time()
    if _left < DEADLINE_SECONDS + MANIFEST_RESERVE_S + 300:
        raise RuntimeError("the exchange URLs expire before this job's hard stop (%.0f s left); "
                           "ask Claude to re-sign them before running" % _left)
    if not INPUT_ZIP.is_file():
        with urllib.request.urlopen(EXCHANGE["inputs"]["url"], timeout=600) as _r:
            if not 200 <= _r.status < 300:
                raise RuntimeError("inputs.zip download failed: HTTP %d" % _r.status)
            _data = _r.read()
        _sha = hashlib.sha256(_data).hexdigest()
        if _sha != INPUT_ZIP_SHA256 or _sha != EXCHANGE["inputs"]["sha256"]:
            raise RuntimeError("downloaded inputs.zip sha256 mismatch: " + _sha + " (not kept)")
        INPUT_ZIP.write_bytes(_data)
        print("inputs.zip downloaded from the exchange:", len(_data), "bytes")
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Stills: FLUX.1-schnell, 4 steps. The exact revision is resolved now and recorded.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os, gc, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers",
                "accelerate", "sentencepiece", "protobuf", "bitsandbytes"], check=True)
import torch
from diffusers import BitsAndBytesConfig, FluxPipeline, FluxTransformer2DModel
from huggingface_hub import HfApi

revision = HfApi().model_info(MODEL_REPO).sha
# Two phases so the text encoders (~10 GB) and the transformer never share the GPU.
# The transformer is ~24 GB in bf16, more than an L4 holds (22 GB): below 30 GB of VRAM
# it loads 8-bit (~12 GB, near-lossless); an A100/H100 keeps bf16.
gpu = torch.cuda.get_device_properties(0)
QUANT = "bf16" if gpu.total_memory >= 30 * 2**30 else "int8"
print("GPU", gpu.name, "%.1f GiB;" % (gpu.total_memory / 2**30), "transformer", QUANT)
if QUANT == "int8":
    # bitsandbytes logs "inputs will be cast from torch.bfloat16 to float16" on every 8-bit
    # matmul (thousands of lines per still). Say it once here, then silence that logger.
    import logging
    logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
    print("8-bit transformer: bitsandbytes casts its matmul inputs to float16 (notice shown once)")
T5_MAX = 256  # FLUX.1-schnell's T5 limit


def look_first(prompt):
    """CLIP reads only 77 tokens and T5 256: put the house style first so it is never cut."""
    k = prompt.lower().find("earnalism house style")
    if k <= 0:
        return prompt
    return prompt[k:].rstrip() + " " + prompt[:k].rstrip(" ,;")


# Phase 1: encode every prompt with only the text encoders on the GPU.
enc = FluxPipeline.from_pretrained(MODEL_REPO, revision=revision, transformer=None, vae=None,
                                   torch_dtype=torch.bfloat16).to("cuda")
# JOB["items"] already has one entry per image: a shot with candidates is <key>--c1 ... --cN
# (seeds seed .. seed+N-1, one shared prompt), so the candidates are encoded once.
EMBEDS, t5_cut, by_text = {}, [], {}
with torch.no_grad():
    for item in JOB["items"]:
        text = look_first(item["prompt"])
        if text not in by_text:
            if len(enc.tokenizer_2(text).input_ids) > T5_MAX:
                t5_cut.append(item["key"])
            pe, pooled, _ = enc.encode_prompt(prompt=text, prompt_2=text, device="cuda",
                                              max_sequence_length=T5_MAX)
            by_text[text] = (pe.cpu(), pooled.cpu())
        EMBEDS[item["key"]] = by_text[text]
print("encoded", len(EMBEDS), "prompts; T5 tail cut on", len(t5_cut), t5_cut)
del enc
gc.collect()
torch.cuda.empty_cache()

# Phase 2: the transformer (and the small VAE) alone on the GPU.
kw = {"revision": revision, "torch_dtype": torch.bfloat16}
if QUANT == "int8":
    kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
transformer = FluxTransformer2DModel.from_pretrained(MODEL_REPO, subfolder="transformer", **kw)
if QUANT == "bf16":
    transformer.to("cuda")
assert next(transformer.parameters()).device.type == "cuda", "transformer is not on the GPU"
pipe = FluxPipeline.from_pretrained(MODEL_REPO, revision=revision, transformer=transformer,
                                    text_encoder=None, text_encoder_2=None, tokenizer=None,
                                    tokenizer_2=None, torch_dtype=torch.bfloat16)
pipe.vae.to("cuda")
print("GPU memory after load: %.1f GiB" % (torch.cuda.memory_allocated() / 2**30))
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "steps": STEPS, "gpu": gpu.name,
              "transformer_precision": QUANT, "prompt_order": "house style first",
              "t5_truncated": t5_cut}
EXPECTED = [item["key"] + ".png" for item in JOB["items"]]
last = 0.0  # seconds the previous still took; the next starts only if it fits (R25)
for item in JOB["items"]:
    if not may_start(last):
        break
    started = time.time()
    gen = torch.Generator("cpu").manual_seed(item["seed"])
    pe, pooled = EMBEDS[item["key"]]
    with torch.no_grad():
        image = pipe(prompt_embeds=pe.to("cuda"), pooled_prompt_embeds=pooled.to("cuda"),
                     num_inference_steps=STEPS, guidance_scale=0.0, max_sequence_length=T5_MAX,
                     width=item["width"], height=item["height"], generator=gen).images[0]
    image.save(OUT_DIR / (item["key"] + ".png"))
    last = time.time() - started
    print("wrote", item["key"], "in %.1f s" % last)


In [ ]:
# Character-and-period gate, run on every still just written (after the stills; nothing here changes
# an image). Per still it writes <key>.gate.json:
#   {"eyes": open eyes found over all detected faces, "hands_detached": bool, "anachronisms": [labels],
#    "anchor_similarity": DINOv2 cosine of the character's person-box crop to its anchor still (null
#    without an anchor), "pass": bool, "needs_review": bool, "notes": [...]}
# pass is true only when every check ran and found nothing. A model that failed, was refused by the
# licence check or returned nothing sets needs_review, never pass. `pick` refuses a candidate whose gate
# is a certain fail unless --override-fail KEY=reason; needs_review is advisory until the title has a
# gate-calibration.json (references/visual-style.md, Gate).
# The whole gate is guarded: it is skipped when the clock is short or the job INCOMPLETE, and any failure
# inside it leaves the stills, the manifest and the upload untouched (the keys are listed as ungated).
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import gc, hashlib, json, os, pathlib, subprocess, sys, time, urllib.request, zipfile

GATE_LICENCE_ALLOWLIST = ('apache-2.0', 'mit')
GATE_ANCHOR_PASS = 0.55
GATE_SETUP_S = 300
GATE_PER_STILL_S = 6
GATE_LABELS = ["gas cylinder", "air conditioner", "electric cable", "spotlight", "modern eyeglasses",
               "plastic", "utility pole", "modern high-rise building"]
GATE_PERSON_LABELS = ["person", "arm"]       # only used to judge whether a hand belongs to a body
GATE_OWL_FAIL = 0.30        # OWLv2 score at or above which an anachronism label counts as found
GATE_OWL_POSSIBLE = 0.15    # between this and GATE_OWL_FAIL: a person must look (needs_review)
GATE_PERSON_SCORE = 0.20    # OWLv2 score at or above which a person/arm box is used
GATE_EYE_OPEN_EAR = 0.15    # eye aspect ratio at or above which an eye counts as open
GATE_FRONTAL = (0.38, 0.62)  # nose-tip position between the cheek edges: inside = frontal face
GATE_MODELS = {
    "face_landmarker": {"declared": "apache-2.0", "hub": None, "url":
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"},
    "hand_landmarker": {"declared": "apache-2.0", "hub": None, "url":
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"},
    "owlv2": {"declared": "apache-2.0", "hub": "google/owlv2-base-patch16-ensemble"},
    "dinov2": {"declared": "apache-2.0", "hub": "facebook/dinov2-small"},
}


# --- gate helper begin (plain functions on plain data; the tests exec this block alone) ---
_EYES = ((33, 160, 158, 133, 153, 144), (362, 385, 387, 263, 373, 380))   # MediaPipe 478-point indices
_NOSE, _CHEEK_L, _CHEEK_R = 1, 234, 454


def _dist(a, b):
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5


def eye_aspect(lm, ids):
    p1, p2, p3, p4, p5, p6 = (lm[i] for i in ids)
    width = _dist(p1, p4)
    return (_dist(p2, p6) + _dist(p3, p5)) / (2 * width) if width > 0 else 0.0


def face_report(lm):
    """lm: pixel (x, y) landmarks of one face. eyes = eyes whose aspect ratio says open (this measures
    openness, not a malformed eye: a closed frontal eye counts as not visible); frontal = the nose tip sits
    between GATE_FRONTAL of the way across the cheek edges."""
    eyes = sum(1 for ids in _EYES if eye_aspect(lm, ids) >= GATE_EYE_OPEN_EAR)
    span = lm[_CHEEK_R][0] - lm[_CHEEK_L][0]
    ratio = (lm[_NOSE][0] - lm[_CHEEK_L][0]) / span if span > 0 else 0.5
    return {"eyes": eyes, "frontal": GATE_FRONTAL[0] <= ratio <= GATE_FRONTAL[1]}


def synthetic_face(open_eyes, yaw):
    """A 478-point test face: cheeks 200 px apart, nose at `yaw` of the way across, open_eyes eyes open."""
    lm = [(0.0, 0.0)] * 478
    lm[_CHEEK_L], lm[_CHEEK_R], lm[_NOSE] = (100.0, 100.0), (300.0, 100.0), (100.0 + 200.0 * yaw, 120.0)
    for n, ids in enumerate(_EYES):
        h = 8.0 if n < open_eyes else 1.0
        x = 140.0 + 80.0 * n
        for i, (dx, dy) in zip(ids, ((0, 0), (10, -h), (30, -h), (40, 0), (30, h), (10, h))):
            lm[i] = (x + dx, 100.0 + dy)
    return lm


def hand_detached(wrist, persons, faces, diag):
    """True: a hand whose wrist is outside every person/arm box (grown by 3% of the image diagonal) while
    something was detected to judge against. False: the wrist lies on a body. None: no person, arm or face
    was detected, so nothing says whether the hand is attached (needs_review)."""
    if not persons and not faces:
        return None
    pad = 0.03 * diag
    for x0, y0, x1, y1 in list(persons) + list(faces):
        if x0 - pad <= wrist[0] <= x1 + pad and y0 - pad <= wrist[1] <= y1 + pad:
            return False
    return True


def crop_box(persons, size):
    """The largest person box clipped to the image (x0, y0, x1, y1 ints), or None if there is none (or it is
    under 16 px): the anchor comparison then uses the whole frame."""
    best = None
    for x0, y0, x1, y1 in persons:
        x0, y0, x1, y1 = max(0, int(x0)), max(0, int(y0)), min(size[0], int(x1)), min(size[1], int(y1))
        if x1 - x0 >= 16 and y1 - y0 >= 16 and (best is None or (x1 - x0) * (y1 - y0) > (best[2] - best[0]) * (best[3] - best[1])):
            best = (x0, y0, x1, y1)
    return best


def gate_licence(name, card_licence, declared, card_read=True):
    """(licence, note), failing closed: a Hub card that could not be read (card_read False) gives (None, why)
    and the model is not loaded. The declared licence stands in only for a card that was read and carries no
    licence field, or for a model with no Hub card. (None, why) too if it is off GATE_LICENCE_ALLOWLIST."""
    if not card_read:
        return None, name + ": model card could not be read; model not loaded"
    card = str(card_licence).strip().lower() if card_licence else None
    lic = card or (str(declared).strip().lower() if declared else None)
    if lic is None:
        return None, name + ": no licence known; model not run"
    if lic not in GATE_LICENCE_ALLOWLIST:
        return None, name + ": licence " + repr(lic) + " is not on the allowlist; model not run"
    return lic, (None if card else name + ": card carries no licence; declared " + lic + " used")


def gate_plan(left_s, incomplete, n_stills, setup_s, per_still_s, reserve_s):
    """(run, why): the whole gate runs only if the job is not INCOMPLETE and the seconds left cover the
    set-up, every still and the manifest reserve. Otherwise it is skipped and the stills stay ungated."""
    if incomplete:
        return False, "the job is INCOMPLETE (a hard stop was reached)"
    need = setup_s + per_still_s * n_stills + reserve_s
    if left_s < need:
        return False, "%.0f s left, the gate needs about %.0f s plus the manifest reserve" % (left_s, need)
    return True, None


def owl_results(res, texts):
    """[(score, label, [x0, y0, x1, y1])] from one OWLv2 post-processed result, whichever transformers
    version made it: labels as text (text_labels, or label strings) or as indices into texts."""
    names = res.get("text_labels")
    if names is None:
        names = list(res["labels"])
    out = []
    for score, name, box in zip(res["scores"], names, res["boxes"]):
        label = name if isinstance(name, str) else texts[int(name)]
        out.append((float(score), label, [float(v) for v in box]))
    return out


def owl_detect(processor, outputs, target_sizes, threshold, texts):
    """OWLv2 post-processing under either transformers name: post_process_grounded_object_detection (newer)
    or post_process_object_detection (older)."""
    grounded = getattr(processor, "post_process_grounded_object_detection", None)
    if grounded is not None:
        res = grounded(outputs=outputs, threshold=threshold, target_sizes=target_sizes, text_labels=[texts])[0]
    else:
        legacy = getattr(processor, "post_process_object_detection", None)
        if legacy is None:
            raise AttributeError("the OWLv2 processor has neither post_process_grounded_object_detection "
                                 "nor post_process_object_detection")
        res = legacy(outputs=outputs, threshold=threshold, target_sizes=target_sizes)[0]
    return owl_results(res, texts)


def decide(faces, hands, anachronisms, possible, similarity, anchor, errors=None):
    """The verdict. faces: [{"eyes", "frontal"}] | None (check did not run); hands: [True|False|None] | None;
    anachronisms / possible: [labels] | None; similarity: float | None; anchor: the still has an anchor image;
    errors: {"faces"|"hands"|"owl"|"anchor": why the check did not run}. An empty faces or hands list is a
    valid result (nothing detected), not an error. fail = a certain defect (needs_review is then false);
    needs_review = something unclear; pass only if neither."""
    errors = errors or {}
    fail, review, notes = False, False, []
    eyes_total = 0
    if faces is None:
        review = True
        notes.append("face check did not run: " + errors.get("faces", "no reason recorded"))
    else:
        for f in faces:
            eyes_total += f["eyes"]
            if f["eyes"] != 2:
                if f["frontal"]:
                    fail = True
                    notes.append("frontal face with %d visible eye(s)" % f["eyes"])
                else:
                    review = True
                    notes.append("profile face with %d visible eye(s)" % f["eyes"])
        if not faces and anchor:
            review = True
            notes.append("a character is expected but no face was detected")
    detached = False
    if hands is None:
        review = True
        notes.append("hand check did not run: " + errors.get("hands", "no reason recorded"))
    else:
        if any(h is True for h in hands):
            detached = True
            fail = True
            notes.append("a hand with no body or arm near its wrist")
        if any(h is None for h in hands):
            review = True
            notes.append("a hand could not be judged attached or detached")
    if anachronisms is None or possible is None:
        review = True
        notes.append("period check did not run: " + errors.get("owl", "no reason recorded"))
        anachronisms = anachronisms or []
    else:
        if anachronisms:
            fail = True
            notes.append("anachronism: " + ", ".join(anachronisms))
        if possible:
            review = True
            notes.append("possible anachronism: " + ", ".join(possible))
    if anchor:
        if similarity is None:
            review = True
            notes.append("anchor similarity did not run: " + errors.get("anchor", "no reason recorded"))
        elif similarity < GATE_ANCHOR_PASS:
            review = True
            notes.append("anchor similarity %.2f is below %.2f (uncalibrated)" % (similarity, GATE_ANCHOR_PASS))
    return {"eyes": eyes_total, "hands_detached": detached, "anachronisms": list(anachronisms),
            "anchor_similarity": None if similarity is None else round(float(similarity), 4),
            "pass": not fail and not review, "needs_review": review and not fail, "notes": notes}
# --- gate helper end ---


# Self-check of the helpers before any model is loaded (a wrong helper must stop here, not pass stills).
assert face_report(synthetic_face(2, 0.5)) == {"eyes": 2, "frontal": True}
assert face_report(synthetic_face(1, 0.9)) == {"eyes": 1, "frontal": False}
assert hand_detached((900, 300), [(100, 100, 500, 900)], [], 1000.0) is True
assert decide([{"eyes": 2, "frontal": True}], [False], [], [], None, False)["pass"] is True
assert decide([], [], [], [], None, False)["pass"] is True        # nothing detected is a valid result

# The gate's intent is recorded first: if anything below fails, the manifest still says it was meant to run.
GATE_INFO = {"intended": True, "ran": False, "models": {}, "refused": [], "ungated": [], "skipped": None,
             "error": None, "thresholds": {
                 "owl_fail": GATE_OWL_FAIL, "owl_possible": GATE_OWL_POSSIBLE, "person_score": GATE_PERSON_SCORE,
                 "eye_open_ear": GATE_EYE_OPEN_EAR, "frontal": list(GATE_FRONTAL),
                 "anchor_pass": GATE_ANCHOR_PASS, "setup_s": GATE_SETUP_S, "per_still_s": GATE_PER_STILL_S}}
MODEL_INFO["gate"] = GATE_INFO
FACE = HAND = OWL = DINO = None


def _licence_and_revision(name, token):
    """Check one model's licence against the allowlist (failing closed) and return (licence, pinned revision).
    A Hub model loads only at the revision whose card was just read."""
    m = GATE_MODELS[name]
    card, rev, read = None, None, True
    if m["hub"]:
        read = False
        try:
            from huggingface_hub import HfApi
            info = HfApi().model_info(m["hub"], token=token)
            data = getattr(info, "card_data", None) or {}
            card = (data.get("license") if hasattr(data, "get") else getattr(data, "license", None)) or next(
                (t.split(":", 1)[1] for t in (getattr(info, "tags", None) or []) if t.startswith("license:")), None)
            rev, read = info.sha, bool(info.sha)
        except Exception as e:
            print("gate:", name, "card unreadable:", type(e).__name__)
    lic, note = gate_licence(name, card, m["declared"], read)
    print("gate model licence:", name, "->", lic, "(allowlist: %s)" % ", ".join(GATE_LICENCE_ALLOWLIST))
    if note:
        print("gate:", note)
    info = {"licence": lic, "declared_licence": m["declared"], "revision": rev}
    GATE_INFO["models"][name] = info
    if lic is None:
        GATE_INFO["refused"].append(name)
    return lic, rev


def _load_models():
    global FACE, HAND, OWL, DINO
    import numpy as np
    token = os.environ.get("HF_TOKEN")
    lic = {n: _licence_and_revision(n, token) for n in GATE_MODELS}
    gate_dir = pathlib.Path("/content/gate-models")
    gate_dir.mkdir(exist_ok=True)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe"], check=True)
        import mediapipe as mp
        from mediapipe.tasks import python as mpt
        from mediapipe.tasks.python import vision as mpv

        def task_file(name):
            path = gate_dir / (name + ".task")
            if not path.is_file():
                urllib.request.urlretrieve(GATE_MODELS[name]["url"], path)
            GATE_INFO["models"][name]["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
            return str(path)
        if lic["face_landmarker"][0]:
            FACE = mpv.FaceLandmarker.create_from_options(mpv.FaceLandmarkerOptions(
                base_options=mpt.BaseOptions(model_asset_path=task_file("face_landmarker")), num_faces=6))
        if lic["hand_landmarker"][0]:
            HAND = mpv.HandLandmarker.create_from_options(mpv.HandLandmarkerOptions(
                base_options=mpt.BaseOptions(model_asset_path=task_file("hand_landmarker")), num_hands=6))
        globals()["mp"] = mp
    except Exception as e:
        print("gate: MediaPipe unavailable:", type(e).__name__, str(e)[:200])
    try:
        import torch
        from transformers import AutoImageProcessor, AutoModel, Owlv2ForObjectDetection, Owlv2Processor
        if lic["owlv2"][0]:
            repo, rev = GATE_MODELS["owlv2"]["hub"], lic["owlv2"][1]
            OWL = (Owlv2Processor.from_pretrained(repo, revision=rev),
                   Owlv2ForObjectDetection.from_pretrained(repo, revision=rev).to("cuda").eval())
        if lic["dinov2"][0]:
            repo, rev = GATE_MODELS["dinov2"]["hub"], lic["dinov2"][1]
            DINO = (AutoImageProcessor.from_pretrained(repo, revision=rev),
                    AutoModel.from_pretrained(repo, revision=rev).to("cuda").eval())
    except Exception as e:
        print("gate: transformers models unavailable:", type(e).__name__, str(e)[:200])
    print("gate models ready: face", FACE is not None, "hand", HAND is not None, "owlv2", OWL is not None,
          "dinov2", DINO is not None)


def _detect_faces(img):
    """(faces, boxes); faces is None when the model is not loaded (an empty list is a valid result)."""
    import numpy as np
    if FACE is None:
        return None, []
    res = FACE.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(img)))
    w, h = img.size
    pts = [[(p.x * w, p.y * h) for p in face] for face in res.face_landmarks]
    return [face_report(lm) for lm in pts], [(min(x for x, _ in lm), min(y for _, y in lm),
                                               max(x for x, _ in lm), max(y for _, y in lm)) for lm in pts]


def _detect_owl(img):
    """(anachronisms, possible, person boxes); None x3 when the model is not loaded."""
    import torch
    if OWL is None:
        return None, None, None
    proc, model = OWL
    texts = GATE_LABELS + GATE_PERSON_LABELS
    inputs = proc(text=[texts], images=img, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model(**inputs)
    side = max(img.size)  # OWLv2 pads the image to a square
    found, possible, persons = set(), set(), []
    for score, text, box in owl_detect(proc, out, torch.tensor([[side, side]]), GATE_OWL_POSSIBLE, texts):
        if text in GATE_PERSON_LABELS:
            if score >= GATE_PERSON_SCORE:
                persons.append(tuple(box))
        elif score >= GATE_OWL_FAIL:
            found.add(text)
        else:
            possible.add(text)
    return sorted(found), sorted(possible - found), persons


def _detect_hands(img, persons, faces):
    """[True|False|None] per hand ([] = no hands, a valid result); None when it could not run."""
    import numpy as np
    if HAND is None:
        raise RuntimeError("hand model not loaded")
    if persons is None:
        raise RuntimeError("needs the person boxes of the period check, which did not run")
    res = HAND.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(img)))
    w, h = img.size
    diag = (w * w + h * h) ** 0.5
    return [hand_detached((lm[0].x * w, lm[0].y * h), persons, faces, diag) for lm in res.hand_landmarks]


_anchor_cache = {}


def _embed(img):
    import torch
    proc, model = DINO
    with torch.no_grad():
        f = model(**proc(images=img, return_tensors="pt").to("cuda")).pooler_output
    return (f / f.norm(dim=-1, keepdim=True))[0].cpu()


def _similarity(img, anchor_name, zf, persons):
    """DINOv2 cosine of the largest person-box crop (the whole frame if there is none) to the anchor still."""
    from PIL import Image
    if DINO is None:
        raise RuntimeError("DINOv2 not loaded")
    if anchor_name not in _anchor_cache:
        _anchor_cache[anchor_name] = _embed(Image.open(zf.open(anchor_name)).convert("RGB"))
    box = crop_box(persons or [], img.size)
    return float((_embed(img.crop(box) if box else img) * _anchor_cache[anchor_name]).sum())


def _gate_time_ok(last_s):
    """Like may_start, but running out of time only leaves the rest ungated: it never marks the job
    INCOMPLETE (the stills themselves are finished)."""
    return DEADLINE_SECONDS - (time.time() - JOB_STARTED) >= last_s + MANIFEST_RESERVE_S


def run_gate():
    from PIL import Image
    t0 = time.time()
    keys = [i["key"] for i in JOB["items"] if (OUT_DIR / (i["key"] + ".png")).is_file()]
    run, why = gate_plan(DEADLINE_SECONDS - (time.time() - JOB_STARTED), INCOMPLETE, len(keys),
                         GATE_SETUP_S, GATE_PER_STILL_S, MANIFEST_RESERVE_S)
    if not run:
        GATE_INFO["skipped"] = why
        print("gate skipped:", why)
        return
    for n in ("pipe", "transformer", "enc"):   # the stills are done: free the GPU for the gate models
        globals().pop(n, None)
    gc.collect()
    import torch
    torch.cuda.empty_cache()
    _load_models()
    last = 0.0
    with zipfile.ZipFile(INPUT_ZIP) as z:
        for item in JOB["items"]:
            png = OUT_DIR / (item["key"] + ".png")
            if not png.is_file() or not _gate_time_ok(last):
                continue
            t = time.time()
            img = Image.open(png).convert("RGB")
            faces = hands = anach = poss = sim = persons = None
            boxes, errors = [], {}
            for name, fn in (("faces", lambda: _detect_faces(img)), ("owl", lambda: _detect_owl(img))):
                try:
                    got = fn()
                    if name == "faces":
                        faces, boxes = got
                        if faces is None:
                            errors["faces"] = "model not loaded"
                    else:
                        anach, poss, persons = got
                        if anach is None:
                            errors["owl"] = "model not loaded"
                except Exception as e:
                    errors[name] = type(e).__name__ + ": " + str(e)[:120]
                    print("gate:", item["key"], name, "failed:", type(e).__name__, str(e)[:200])
            try:
                hands = _detect_hands(img, persons, boxes)
            except Exception as e:
                errors["hands"] = type(e).__name__ + ": " + str(e)[:120]
            if item.get("anchor"):
                try:
                    sim = _similarity(img, item["anchor"], z, persons)
                except Exception as e:
                    errors["anchor"] = type(e).__name__ + ": " + str(e)[:120]
            verdict = decide(faces, hands, anach, poss, sim, bool(item.get("anchor")), errors)
            if item.get("anchor") and sim is not None and not crop_box(persons or [], img.size):
                verdict["notes"].append("no person box: similarity used the whole frame")
            (OUT_DIR / (item["key"] + ".gate.json")).write_text(json.dumps(verdict, indent=1, sort_keys=True) + "\n")
            last = time.time() - t
            print("gate", item["key"], "pass" if verdict["pass"] else ("REVIEW" if verdict["needs_review"] else "FAIL"),
                  "; ".join(verdict["notes"]))
    GATE_INFO["ran"] = True
    GATE_INFO["seconds"] = round(time.time() - t0, 1)


try:
    run_gate()
except Exception as _e:   # nothing in the gate may stop the manifest and upload
    GATE_INFO["error"] = type(_e).__name__ + ": " + str(_e)[:200]
    print("gate failed, stills untouched:", GATE_INFO["error"])
GATE_INFO["ungated"] = sorted(i["key"] for i in JOB["items"] if (OUT_DIR / (i["key"] + ".png")).is_file()
                              and not (OUT_DIR / (i["key"] + ".gate.json")).is_file())
if GATE_INFO["ungated"]:
    print("gate: ungated stills:", ", ".join(GATE_INFO["ungated"]))


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if (p.is_file() and p.name not in ("manifest.json", "inputs.zip", "exchange.json", "done.json", "done.json.tmp")
            and p.suffix != ".ipynb"):  # Colab keeps the job notebook in this folder
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
if globals().get("EXCHANGE") is not None:  # R2 exchange: one uncompressed tar, PUT once
    import os, tarfile, tempfile, urllib.error, urllib.request
    _fd, _tar = tempfile.mkstemp(suffix=".tar")  # local disk, never the Drive job folder
    os.close(_fd)
    with tarfile.open(_tar, "w", format=tarfile.PAX_FORMAT) as _t:
        for _name in ["manifest.json"] + sorted(outputs):
            _t.add(str(OUT_DIR / _name), arcname=_name, recursive=False)
    _size = os.path.getsize(_tar)
    _headers = dict(EXCHANGE["outputs"]["headers"], **{"Content-Length": str(_size)})
    with open(_tar, "rb") as _body:
        _req = urllib.request.Request(EXCHANGE["outputs"]["url"], data=_body, method="PUT",
                                      headers=_headers)
        try:
            with urllib.request.urlopen(_req, timeout=1800) as _r:
                _status = _r.status
        except urllib.error.HTTPError as _e:
            _status = _e.code
    os.unlink(_tar)
    print("outputs.tar upload: HTTP", _status, "(%d bytes)" % _size)
    if not 200 <= _status < 300:
        raise RuntimeError("outputs.tar upload failed: HTTP %d; the outputs stay in Drive, tell Claude"
                           % _status)
    print("outputs uploaded to the exchange; tell Claude the CU Colab shows")
